In [ ]:
!pip install torch numpy transformers datasets rouge_score bert-score pandas matplotlib tqdm causal-conv1d mamba-ssm

In [ ]:
import time
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GenerationConfig,
    MambaConfig,
    MambaForCausalLM
)
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import pandas as pd
import random
import matplotlib.pyplot as plt
from tqdm import tqdm
from mamba_ssm.models.mixer_seq_simple import MambaLMHeadModel

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

max_input_length = 1024
max_new_tokens = 512
batch_size = 1
num_samples = 50

models_to_compare = {
    "Transformer": {
        "model_name": "nicholasKluge/Aira-OPT-350M",
        "tokenizer_name": "nicholasKluge/Aira-OPT-350M",
    },
    "Mamba": {
        "model_name": "OuteAI/Lite-Oute-2-Mamba2Attn-250M-Instruct",
        "tokenizer_name": "OuteAI/Lite-Oute-2-Mamba2Attn-250M-Instruct",
    }
}


Using device: cuda


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_and_tokenizer(model_config):
    tokenizer = AutoTokenizer.from_pretrained(model_config["tokenizer_name"])

    model = AutoModelForCausalLM.from_pretrained(
        model_config["model_name"],
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        trust_remote_code=True
    )

    model.to(device)
    model.eval()
    return model, tokenizer


In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens=512):
    if isinstance(model, MambaLMHeadModel):
        # Specific prompt style for the Mamba-2 model
        system_prompt = "You are a helpful assistant."

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]

        input_ids = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)

        # Typical hyperparameters for this model
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "temperature": 0.1,
            "repetition_penalty": 1.12,
            "do_sample": True,
        }
    else:
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=max_input_length).to(device)

        # Typical hyperparameters for this model
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": True,
            "temperature": 0.5,
            "top_p": 0.6,
            "top_k": 30,
            "repetition_penalty": 1.2,
            "pad_token_id": tokenizer.eos_token_id,
            "attention_mask": inputs["attention_mask"],
        }
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            **generation_kwargs
        )
    end_time = time.time()

    generation_time = end_time - start_time
    generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    return generated_text, generation_time

In [ ]:
def rouge_evaluate_generations(references, generations):
    """
    Evaluate generations using ROUGE.

    Args:
        references: List of reference texts
        generations: List of generated texts

    Returns:
        Dictionary with ROUGE-1, ROUGE-2, and ROUGE-L scores
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    scores = {
        'rouge1': [],
        'rouge2': [],
        'rougeL': []
    }

    for ref, gen in zip(references, generations):
        results = scorer.score(ref, gen)
        for metric in scores.keys():
            scores[metric].append(results[metric].fmeasure)

    mean_scores = {metric: np.mean(values) for metric, values in scores.items()}
    return mean_scores

def bert_score_evaluate(references, generations, lang="en", verbose=False):
    """
    Evaluate generations using BERTScore.

    Args:
        references: List of reference texts
        generations: List of generated texts
        lang: Language of the texts (default: "en")
        verbose: Whether to print progress (default: False)

    Returns:
        Dictionary with precision, recall, and F1 scores
    """
    P, R, F1 = bert_score(generations, references, lang=lang, verbose=verbose)

    P_list = P.cpu().numpy().tolist()
    R_list = R.cpu().numpy().tolist()
    F1_list = F1.cpu().numpy().tolist()

    mean_scores = {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "F1": F1.mean().item()
    }

    individual_scores = {
        "precision": P_list,
        "recall": R_list,
        "F1": F1_list
    }

    return mean_scores, individual_scores

In [ ]:
# Load the models
model_tokenizers = {}
for model_name, model_config in models_to_compare.items():
    print(f"Loading {model_name} model...")
    model, tokenizer = load_model_and_tokenizer(model_config)

    model_tokenizers[model_name] = {"model": model, "tokenizer": tokenizer}

Loading Transformer model...
Loading Mamba model...


In [ ]:
# Load the XSum dataset
print("Loading XSum dataset...")
dataset = load_dataset("xsum", split="validation")

input_column = "document"
target_column = "summary"
prompt_template = "Summarize the following:\n{}\n\nSummary:"

print(dataset[0][input_column])
print(dataset[0][target_column])

# Select a subset as our eval dataset
eval_dataset = dataset.select(range(min(num_samples, len(dataset))))

Loading XSum dataset...
The ex-Reading defender denied fraudulent trading charges relating to the Sodje Sports Foundation - a charity to raise money for Nigerian sport.
Mr Sodje, 37, is jointly charged with elder brothers Efe, 44, Bright, 50 and Stephen, 42.
Appearing at the Old Bailey earlier, all four denied the offence.
The charge relates to offences which allegedly took place between 2008 and 2014.
Sam, from Kent, Efe and Bright, of Greater Manchester, and Stephen, from Bexley, are due to stand trial in July.
They were all released on bail.
Former Premier League footballer Sam Sodje has appeared in court alongside three brothers accused of charity fraud.


In [ ]:
results = {}

for model_name, model_config in models_to_compare.items():
    model = model_tokenizers[model_name]["model"]
    tokenizer = model_tokenizers[model_name]["tokenizer"]

    generations = []
    generation_times = []
    references = []
    input_lengths = []
    output_lengths = []

    for sample in tqdm(eval_dataset):
        input_text = sample[input_column]
        reference = sample[target_column]

        # Format into prompt template from above
        prompt = prompt_template.format(input_text)

        input_tokens = tokenizer(input_text, return_tensors="pt").input_ids.shape[1]
        input_lengths.append(input_tokens)

        generated_text, gen_time = generate_text(
            model, tokenizer, prompt, max_new_tokens=max_new_tokens
        )

        # Get output length
        output_tokens = tokenizer(generated_text, return_tensors="pt").input_ids.shape[1]
        output_lengths.append(output_tokens)

        generations.append(generated_text)
        generation_times.append(gen_time)
        references.append(reference)

    rouge_scores = rouge_evaluate_generations(
        references,
        generations
    )

    bertscore_means, bertscore_individuals = bert_score_evaluate(
        references,
        generations,
        lang="en",
        verbose=True
    )

    # Determine extent of compression for summarization.
    avg_compression_ratio = np.mean([i/o if o > 0 else 0 for i, o in zip(input_lengths, output_lengths)])

    results[model_name] = {
        "rouge_scores": rouge_scores,
        "bertscore_means": bertscore_means,
        "bertscore_individuals": bertscore_individuals,
        "avg_generation_time": np.mean(generation_times),
        "avg_compression_ratio": avg_compression_ratio,
        "generation_times": generation_times,
        "generations": generations,
        "references": references,
        "input_lengths": input_lengths,
        "output_lengths": output_lengths
    }

    del model
    torch.cuda.empty_cache()


100%|██████████| 50/50 [00:34<00:00,  1.43it/s]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/2 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.40 seconds, 35.66 sentences/sec


In [ ]:
print("\n----- EVALUATION RESULTS -----")
for model_name, result in results.items():
    print(f"\n{model_name} Results:")
    print(f"Average Generation Time: {result['avg_generation_time']:.4f} seconds")
    print(f"Average Compression Ratio: {result['avg_compression_ratio']:.2f}x")

    print("ROUGE Scores:")
    for metric, score in result['rouge_scores'].items():
        print(f"  {metric}: {score:.4f}")

    print("BERTScore:")
    for metric, score in result['bertscore_means'].items():
        print(f"  {metric}: {score:.4f}")


----- EVALUATION RESULTS -----

Transformer Results:
Average Generation Time: 0.6927 seconds
Average Compression Ratio: 14.94x
ROUGE Scores:
  rouge1: 0.1640
  rouge2: 0.0288
  rougeL: 0.1258
BERTScore:
  precision: 0.8537
  recall: 0.8590
  F1: 0.8562

Mamba Results:
Average Generation Time: 2.0122 seconds
Average Compression Ratio: 8.64x
ROUGE Scores:
  rouge1: 0.1556
  rouge2: 0.0196
  rougeL: 0.1160
BERTScore:
  precision: 0.8472
  recall: 0.8611
  F1: 0.8539


In [ ]:
from operator import itemgetter

reference_model = "Transformer"
top_n = 3

# Compute all ROUGE-1 scores
scored_examples = []
for i in range(len(eval_dataset)):
    reference = results[reference_model]["references"][i]
    generation = results[reference_model]["generations"][i]

    rouge1 = rouge_scorer.RougeScorer(['rouge1']).score(reference, generation)["rouge1"].fmeasure
    scored_examples.append((i, rouge1))

# Get top N examples by score
top_indices = [idx for idx, _ in sorted(scored_examples, key=itemgetter(1), reverse=True)[:top_n]]


print("\n----- EXAMPLE GENERATIONS -----")

for i, idx in enumerate(top_indices):
    print(f"\n\n=== Example {i+1} ===")

    # Display the input text (truncated for readability)
    input_text = eval_dataset[i][input_column]
    print(f"\nInput text (truncated):\n{input_text[:300]}...")

    # Display the reference summary
    reference = eval_dataset[i][target_column]
    print(f"\nReference summary:\n{reference}")

    # Display generations from each model
    for model_name in models_to_compare.keys():
        generation = results[model_name]["generations"][i]
        rouge1_score = rouge_scorer.RougeScorer(['rouge1']).score(reference, generation)["rouge1"].fmeasure

        print(f"\n{model_name} generation (ROUGE-1: {rouge1_score:.4f}):")
        print(f"{generation}")

        # Display generation stats
        gen_time = results[model_name]["generation_times"][i]
        input_len = results[model_name]["input_lengths"][i]
        output_len = results[model_name]["output_lengths"][i]

        print(f"Generation time: {gen_time:.2f}s | Input length: {input_len} tokens | "
              f"Output length: {output_len} tokens | Compression ratio: {input_len/output_len if output_len > 0 else 0:.2f}x")

# Create a side-by-side comparison table for a visual comparison
print("\n\n----- SIDE-BY-SIDE COMPARISON -----")

comparison_data = []
for i in range(top_n):
    row = {
        "Example": i+1,
        "Input (truncated)": eval_dataset[i][input_column][:100] + "...",
        "Reference": eval_dataset[i][target_column]
    }

    for model_name in models_to_compare.keys():
        row[f"{model_name} Generation"] = results[model_name]["generations"][i]
        row[f"{model_name} ROUGE-1"] = rouge_scorer.RougeScorer(['rouge1']).score(
            eval_dataset[i][target_column],
            results[model_name]["generations"][i]
        )["rouge1"].fmeasure

    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

# Display the comparison table
pd.set_option('display.max_colwidth', 50)
print(comparison_df)
pd.reset_option('display.max_colwidth')

# Save the comparison to CSV for better viewing
comparison_df.to_csv("model_comparison_examples.csv", index=False)
print("\nDetailed comparison saved to model_comparison_examples.csv")

# Create visualizations comparing the models
plt.figure(figsize=(12, 8))

# Plot 1: Generation Times
plt.subplot(2, 2, 1)
for model_name in models_to_compare.keys():
    plt.plot(results[model_name]["generation_times"], label=model_name)
plt.title("Generation Times per Example")
plt.xlabel("Example Index")
plt.ylabel("Time (seconds)")
plt.legend()

# Plot 2: Output Lengths
plt.subplot(2, 2, 2)
for model_name in models_to_compare.keys():
    plt.plot(results[model_name]["output_lengths"], label=model_name)
plt.title("Output Lengths per Example")
plt.xlabel("Example Index")
plt.ylabel("Tokens")
plt.legend()

# Plot 3: Compression Ratios
plt.subplot(2, 2, 3)
for model_name in models_to_compare.keys():
    compression_ratios = [i/o if o > 0 else 0 for i, o in zip(
        results[model_name]["input_lengths"],
        results[model_name]["output_lengths"]
    )]
    plt.plot(compression_ratios, label=model_name)
plt.title("Compression Ratios per Example")
plt.xlabel("Example Index")
plt.ylabel("Ratio (Input/Output)")
plt.legend()

# Plot 4: ROUGE-1 Scores
plt.subplot(2, 2, 4)
rouge_scores_by_example = {}
for model_name in models_to_compare.keys():
    rouge_scores_by_example[model_name] = []
    for i in range(len(eval_dataset)):
        rouge1 = rouge_scorer.RougeScorer(['rouge1']).score(
            results[model_name]["references"][i],
            results[model_name]["generations"][i]
        )["rouge1"].fmeasure
        rouge_scores_by_example[model_name].append(rouge1)
    plt.plot(rouge_scores_by_example[model_name], label=model_name)
plt.title("ROUGE-1 Scores per Example")
plt.xlabel("Example Index")
plt.ylabel("ROUGE-1 F1")
plt.legend()

plt.tight_layout()
plt.savefig("model_comparison_metrics.png")
plt.close()

print("\nComparison visualizations saved to model_comparison_metrics.png")


----- EXAMPLE GENERATIONS -----


=== Example 1 ===

Input text (truncated):
The ex-Reading defender denied fraudulent trading charges relating to the Sodje Sports Foundation - a charity to raise money for Nigerian sport.
Mr Sodje, 37, is jointly charged with elder brothers Efe, 44, Bright, 50 and Stephen, 42.
Appearing at the Old Bailey earlier, all four denied the offence....

Reference summary:
Former Premier League footballer Sam Sodje has appeared in court alongside three brothers accused of charity fraud.

Transformer generation (ROUGE-1: 0.1053):

Ex- Reading defender denies fraudulent trading charges relating to the sodje Sports Foundation - a charity to raise funds for Nigerian sports.
Generation time: 0.54s | Input length: 122 tokens | Output length: 27 tokens | Compression ratio: 4.52x

Mamba generation (ROUGE-1: 0.1026):
The ex-Reading defender denied fraudulent trading charges related to the Sodje Sports Foundation – a charity to raise money for Nigerian sport.
Generation